## Experiment 2 — Pima Indians Diabetes

### Dataset
The [Pima Indians Diabetes dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) contains 768 observations of female patients aged 21 or older, with 8 clinical features and a binary outcome indicating the presence of type 2 diabetes. The dataset is moderately imbalanced — ~35% positive (diabetes) and ~65% negative.

### Results
Hyperparameter optimization via `XGBOptClf` outperforms the default XGBoost baseline by approximately +0.04 AUC. To this end, we have some evedince that tuning has a meaningful impact on this dataset. 

A pre-processing strategy for clinically implausible zeros (`Glucose`, `BloodPressure`,
`SkinThickness`, `Insulin`, `BMI`) was evaluated:


| Configuration | Test AUC ± std | Dev AUC ± std | Gap |
|---|---|---|---|
| Default XGBoost (no preprocessing) | 0.7914 ± 0.0284 | 100 ± 0 | 0.2086 |
| Default XGBoost (zeros as NaN) | 0.7947 ± 0.0327 | 100 ± 0 | 0.2053 |
| XGBOptClf (no preprocessing) (`std_penalty=0.5`)| 0.8330 ± 0.0257 | 0.8869 ± 0.0141 | 0.0539 |
| XGBOptClf (no preprocessing) (`std_penalty=0.5` `alpha = 0`, `lambda=1`) |0.8293 ± 0.0248 | 0.9031 ± 0.0064| 0.0737 |
| XGBOptClf (zeros as NaN) (`std_penalty=0.5`) | 0.8353 ± 0.0272 |  0.9061 ± 0.0143| 0.0708 |
| XGBOptClf (`std_penalty=2.0`, `alpha=0`, `gamma=0`)(zeros as NaN) (best)** | 0.8407 ± 0.0254 | 0.8963 ± 0.0109 | 0.0556 |

My zero-encoding strategy is consistent with Table (I) in [Pyne et al.](https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=10152382), which identifies clinically implausible zero values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`. The aforementioned study treats zero values in `Pregnancies` as missing, whereas they are left untouched for this study.
To me, no *pregnancy* is clinically valid.

### Nuisance Parameters
Low-importance hyperparameters (`gamma`, `alpha`) were identified via Optuna and fixed at their XGBoost defaults, reducing the search space and improving optimization stability.
This process is legitimate; for example, this [article](https://journals.sagepub.com/doi/10.1191/1471082X04st068oa) suggests a similar strategy in the context of joint modelling. Simply put, primary parameters (e.g., `max_depth`, `min_child_weight`) should be prioritized when datasets are small- or medium-sized.

### Data Analysis
Analyzing data is an iterative process that requires some back-and-forth before achieving some stable results. 

### Conclusions
The results lead to the following conclusions:

- **Zeros encoded as NaN improves performance** — replacing biologically implausible
  zeros with *NaN* consistently improves test AUC.
- **Freezing nuisance parameters has mixed results** — freezing `alpha` and `lambda` at their defaults stabilizes the inner CV (dev std drops from `0.0141` to `0.0064`), 
  but widens the dev/test gap (from `0.054` to `0.074`).
- **Increasing `std_penalty` is the most effective consideration** — `std_penalty=2.0` with
  the `frozen_params={"gamma": 0, "alpha": 0}` setup achieves the best overall result: the highest test AUC (`0.8407`) along with the most hyperparameter selection 
  and the number of tress across folds

- **The default `XGBClassifier` gap is large** — with `100` fixed trees and no hyperparameter tuning, the dev/test gap exceeds `0.20`, indicating severe overfitting
  to the training folds. Early stopping and Optuna optimization reduce this gap to`0.056` in the best configuration.

In [139]:
#np.round((0.8963 - 0.8407), 4)

In [140]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import xgboost as xgb
from src.xgb_opt_clf import XGBOptClf
from src.helper_functions import *
import os
import tempfile
import optuna.visualization as vis
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import json

# Study Setup

In [152]:
N_TRIALS = 100
N_FOLDS = 5
RANDOM_STATE = 42
STUDY_NAME = "xgb_opt_clf_pima"

### Data Load 

- The Pima Indians Diabetes dataset is loaded directly from GitHub into a pandas DataFrame.
- Since the CSV has no header row, column names are supplied manually via `names=cols`.
- `head()` visually confirms the data loaded correctly.
- `shape` verifies the expected dimensions of 768 rows and 9 columns.


In [147]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)
print(df.head())
print(df.shape)  # should be (768, 9)

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


In [148]:
df["Outcome"].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [149]:
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values
columns = list(df.drop("Outcome", axis=1).columns)

In [150]:
columns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

## I) No preprocessing applied
First, the pipeline will be trained on the Pima data without any preprocessing —
zeros are left *unthouched* and passed directly to the model.

In [154]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    datefmt="%H:%M:%S")

import time

logger = logging.getLogger(__name__)

clf_baseline = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS)

start = time.time()
nested_cv_baseline = nested_cv_score(clf_baseline, X, y, n_outer=5)
end = time.time()
logger.info(f"Nested CV took {end - start:.2f} seconds")

00:47:34 — Nested CV: outer fold 1/5
00:47:34 — Starting Optuna optimization for 100 trials
00:47:53 — Nested CV: fold 1 — test AUC = 0.8535, dev AUC = 0.8963, MCC = 0.5067, Brier = 0.1495, threshold = 0.3830
00:47:53 — Nested CV: outer fold 2/5
00:47:53 — Starting Optuna optimization for 100 trials
00:48:15 — Nested CV: fold 2 — test AUC = 0.8726, dev AUC = 0.8691, MCC = 0.4957, Brier = 0.2261, threshold = 0.3491
00:48:15 — Nested CV: outer fold 3/5
00:48:15 — Starting Optuna optimization for 100 trials
00:48:33 — Nested CV: fold 3 — test AUC = 0.8207, dev AUC = 0.8756, MCC = 0.4942, Brier = 0.1605, threshold = 0.3273
00:48:33 — Nested CV: outer fold 4/5
00:48:33 — Starting Optuna optimization for 100 trials
00:48:59 — Nested CV: fold 4 — test AUC = 0.8113, dev AUC = 0.9084, MCC = 0.3835, Brier = 0.2248, threshold = 0.3509
00:48:59 — Nested CV: outer fold 5/5
00:48:59 — Starting Optuna optimization for 100 trials
00:49:21 — Nested CV: fold 5 — test AUC = 0.8068, dev AUC = 0.8849, MCC 

### 
i) Inspecting the number of trees across folds, we can see that the obtained values are low.
This suggests instability since the mean is only around 16 trees.

In [155]:
nested_cv_baseline['n_trees']

[18, 8, 15, 10, 27]

ii)Likewise, it can be noticed that the hyperparameters show significant instability
across folds (i.e., their values vary significantly).

In [156]:
pd.DataFrame(nested_cv_baseline["best_params"])

,lambda,alpha,learning_rate,max_depth,min_child_weight,gamma,subsample,colsample_bytree
0,0.121435,0.687562,0.198808,6,9,0.014296,0.708636,0.654622
1,0.016905,0.054721,0.001776,3,8,0.055931,0.829062,0.630926
2,0.271507,0.556534,0.166978,2,11,0.215736,0.835483,0.856791
3,0.021053,0.023856,0.001685,4,2,0.009301,0.659291,0.745536
4,0.052914,0.001174,0.063145,4,10,0.647362,0.638665,0.649363


From (i) and (ii), let us inspect their importance using Optuna on the entire dataset to identify which hyperparameters are worth tuning and which can be frozen. Using the entire set, namely 
1. Fit on the full dataset; then inspect `param_importances()` so as to decide which hyperparameters to freeze.
2. Run nested CV with `frozen_params`; then obtain  some generalization performance.

In [87]:
clf_inspection = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS)
clf_inspection.fit(X, y)

01:49:34 — Starting Optuna optimization for 100 trials


,n_trials,100
,nfold,5
,random_state,42
,n_jobs,None
,std_penalty,0.5
,eval_metric,'auc'
,use_multivariate,False
,is_stratified,True
,scale_pos_weight,None
,frozen_params,{}


In [138]:
clf_inspection.param_importances()

{'min_child_weight': np.float64(0.5686765947194887),
 'subsample': np.float64(0.1671608462065177),
 'colsample_bytree': np.float64(0.12159158242826998),
 'learning_rate': np.float64(0.1063679806419431),
 'alpha': np.float64(0.015057465924149737),
 'gamma': np.float64(0.012196512562841625),
 'max_depth': np.float64(0.006141883093126008),
 'lambda': np.float64(0.0028071344236630514)}

The results above might be expected, and we can freeze `gamma` and `lambda`
at their XGBoost defaults, while leaving `max_depth` (this is a primary parameter and worth tuning) free for Optuna to tune:

- `gamma = 0`
- `lambda = 1`

In [92]:
clf_frozen = XGBOptClf(n_trials=100, nfold=N_FOLDS, frozen_params={"lambda": 1, "gamma": 0})

start = time.time()

nested_cv_frozen_penalty_imp = nested_cv_score(clf_frozen, X, y, n_outer=5)
end = time.time()
logger.info(f"Nested CV took {end - start:.2f} seconds")

01:54:30 — Nested CV: outer fold 1/5
01:54:30 — Starting Optuna optimization for 100 trials
01:54:46 — Nested CV: fold 1 — test AUC = 0.8498, dev AUC = 0.9002, MCC = 0.4975, Brier = 0.1530, threshold = 0.3401
01:54:46 — Nested CV: outer fold 2/5
01:54:46 — Starting Optuna optimization for 100 trials
01:55:12 — Nested CV: fold 2 — test AUC = 0.8623, dev AUC = 0.9125, MCC = 0.5291, Brier = 0.2232, threshold = 0.3504
01:55:12 — Nested CV: outer fold 3/5
01:55:12 — Starting Optuna optimization for 100 trials
01:55:27 — Nested CV: fold 3 — test AUC = 0.8298, dev AUC = 0.8975, MCC = 0.5344, Brier = 0.1559, threshold = 0.3384
01:55:27 — Nested CV: outer fold 4/5
01:55:27 — Starting Optuna optimization for 100 trials
01:55:46 — Nested CV: fold 4 — test AUC = 0.8100, dev AUC = 0.9089, MCC = 0.4794, Brier = 0.2220, threshold = 0.3516
01:55:46 — Nested CV: outer fold 5/5
01:55:46 — Starting Optuna optimization for 100 trials
01:56:01 — Nested CV: fold 5 — test AUC = 0.7947, dev AUC = 0.8964, MCC 

---

After freezing `gamma` and `lambda` at their defaults, the results are as follows:

- **Mean test AUC:** 0.8293 ± 0.0248
- **Mean dev AUC:** 0.9031 ± 0.0064 The dev std dropped significantly to 0.0064, indicating a much more stable inner CV.

By freezing `gamma=0` and `lambda=1`, Optuna now focuses its trials on the parameters that "move" the objective function. 
This leads to better calibrated learning rates and regularization, allowing the model to grow more trees (29.6) before early stopping kicks in; it is a sign of more stable convergence

In [93]:
nested_cv_frozen_penalty_imp['n_trees']

[22, 8, 23, 43, 27]

In [94]:
pd.DataFrame(nested_cv_frozen_penalty_imp["best_params"])

,alpha,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree
0,0.009754,0.227933,4,8,0.763611,0.562974
1,0.072304,0.004992,5,3,0.827395,0.720811
2,0.141894,0.170878,3,5,0.550115,0.806236
3,0.053533,0.001157,5,3,0.601108,0.661867
4,0.922112,0.002289,4,2,0.517406,0.868148


## II) Zeros encoded as NaN

The `encode_zeros_as_nan` function is leakage-safe. Since the transformation is label-free and does not fit anything on the training data, applying it in advance on the full dataset is valid

In [ ]:
# Helper functions 
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

def encode_zeros_as_nan(X, zero_cols, columns):
    zero_idx = [columns.index(c) for c in zero_cols]
    X = X.astype(float)
    X[:, zero_idx] = np.where(X[:, zero_idx] == 0, np.nan, X[:, zero_idx])
    return X

In [96]:
feature_cols = [c for c in cols if c != 'Outcome']
X_nan = encode_zeros_as_nan(X, zero_cols, feature_cols)

In [134]:
pd.DataFrame(X_nan).head()

,0,1,2,3,4,5,6,7
0,6.0,148.0,72.0,35.0,NaN,33.6,0.627,50.0
1,1.0,85.0,66.0,29.0,NaN,26.6,0.351,31.0
2,8.0,183.0,64.0,NaN,NaN,23.3,0.672,32.0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0


In [135]:
pd.DataFrame(X).head()

,0,1,2,3,4,5,6,7
0,6.0,148.0,72.0,35.0,0.0,33.6,0.627,50.0
1,1.0,85.0,66.0,29.0,0.0,26.6,0.351,31.0
2,8.0,183.0,64.0,0.0,0.0,23.3,0.672,32.0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0


In [54]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    datefmt="%H:%M:%S"
)

clf = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS)

logger = logging.getLogger(__name__)
nested_cv_nan = nested_cv_score(clf, X_nan, y, n_outer=5)

01:13:10 — Nested CV: outer fold 1/5
01:13:10 — Starting Optuna optimization for 100 trials
01:13:32 — Nested CV: fold 1 — test AUC = 0.8600, dev AUC = 0.8965, MCC = 0.5080, Brier = 0.1481, threshold = 0.3383
01:13:32 — Nested CV: outer fold 2/5
01:13:32 — Starting Optuna optimization for 100 trials
01:13:54 — Nested CV: fold 2 — test AUC = 0.8659, dev AUC = 0.9112, MCC = 0.5599, Brier = 0.1437, threshold = 0.3735
01:13:54 — Nested CV: outer fold 3/5
01:13:54 — Starting Optuna optimization for 100 trials
01:14:13 — Nested CV: fold 3 — test AUC = 0.8383, dev AUC = 0.8854, MCC = 0.5570, Brier = 0.1531, threshold = 0.4064
01:14:13 — Nested CV: outer fold 4/5
01:14:13 — Starting Optuna optimization for 100 trials
01:14:39 — Nested CV: fold 4 — test AUC = 0.8211, dev AUC = 0.9097, MCC = 0.5170, Brier = 0.1861, threshold = 0.3524
01:14:39 — Nested CV: outer fold 5/5
01:14:39 — Starting Optuna optimization for 100 trials
01:15:10 — Nested CV: fold 5 — test AUC = 0.7911, dev AUC = 0.9278, MCC 

In [98]:
nested_cv_nan['n_trees']

[15, 34, 34, 61, 17]

In [99]:
clf_inspection_nans = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS)
clf_inspection_nans.fit(X_nan, y)


02:00:28 — Starting Optuna optimization for 100 trials


,n_trials,100
,nfold,5
,random_state,42
,n_jobs,None
,std_penalty,0.5
,eval_metric,'auc'
,use_multivariate,False
,is_stratified,True
,scale_pos_weight,None
,frozen_params,{}


This calls the `param_importances()` method on a classifier fitted on the full dataset with zeros encoded as NaN. It returns the fANOVA-based importance scores for each hyperparameter.

The `_nan` suffix indicates that zeros were replaced with NaN before fitting, meaning the importances reflect the hyperparameter scape on the "cleaned" data.


In [101]:
clf_inspection_nans.param_importances()

{'min_child_weight': np.float64(0.7045547222781012),
 'lambda': np.float64(0.10139497202108713),
 'subsample': np.float64(0.06365037146182217),
 'max_depth': np.float64(0.055328685600009815),
 'colsample_bytree': np.float64(0.0387717501548906),
 'alpha': np.float64(0.030710089016063934),
 'learning_rate': np.float64(0.0044530816884377515),
 'gamma': np.float64(0.0011363277795871925)}

In [102]:
clf_frozen_nan = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS, frozen_params={"gamma": 0, "alpha": 0})

logger = logging.getLogger(__name__)
nested_cv_frozen_nan = nested_cv_score(clf_frozen_nan, X_nan, y, n_outer=5)

02:02:56 — Nested CV: outer fold 1/5
02:02:56 — Starting Optuna optimization for 100 trials
02:03:14 — Nested CV: fold 1 — test AUC = 0.8531, dev AUC = 0.8916, MCC = 0.5234, Brier = 0.1701, threshold = 0.3716
02:03:14 — Nested CV: outer fold 2/5
02:03:14 — Starting Optuna optimization for 100 trials
02:03:37 — Nested CV: fold 2 — test AUC = 0.8661, dev AUC = 0.9072, MCC = 0.5344, Brier = 0.2176, threshold = 0.3508
02:03:37 — Nested CV: outer fold 3/5
02:03:37 — Starting Optuna optimization for 100 trials
02:03:56 — Nested CV: fold 3 — test AUC = 0.8426, dev AUC = 0.9115, MCC = 0.4975, Brier = 0.1521, threshold = 0.3364
02:03:56 — Nested CV: outer fold 4/5
02:03:56 — Starting Optuna optimization for 100 trials
02:04:19 — Nested CV: fold 4 — test AUC = 0.8219, dev AUC = 0.9522, MCC = 0.4996, Brier = 0.1649, threshold = 0.3536
02:04:19 — Nested CV: outer fold 5/5
02:04:19 — Starting Optuna optimization for 100 trials
02:04:40 — Nested CV: fold 5 — test AUC = 0.7992, dev AUC = 0.9266, MCC 

Mean is 36.4 -- the highest mean so ths, which is a good sign. But `fold 4` (63) is again atypical compared to the remaining folds. Namely, the remaining folds are quite consistent around 32–35 though, which is encouraging.

In [104]:
nested_cv_frozen_nan['n_trees']

[32, 35, 35, 63, 17]

Building on the previous configuration, `std_penalty` is increased from `0.5` to `1.0` to more aggressively penalize unstable trials, while keeping `gamma` and `alpha` fixed/frozen 
at their XGBoost defaults. The goal is to close the dev/test gap observed in the previous run while preserving the test AUC gains.

In [105]:
clf_frozen_nan_high_penalty = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS, frozen_params={"gamma": 0, "alpha": 0}, std_penalty=1.0)

logger = logging.getLogger(__name__)
nested_cv_frozen_nan_high_penatly = nested_cv_score(clf_frozen_nan_high_penalty, X_nan, y, n_outer=5)

02:07:17 — Nested CV: outer fold 1/5
02:07:17 — Starting Optuna optimization for 100 trials
02:07:35 — Nested CV: fold 1 — test AUC = 0.8588, dev AUC = 0.8833, MCC = 0.5462, Brier = 0.1667, threshold = 0.3495
02:07:35 — Nested CV: outer fold 2/5
02:07:35 — Starting Optuna optimization for 100 trials
02:07:58 — Nested CV: fold 2 — test AUC = 0.8789, dev AUC = 0.8864, MCC = 0.5817, Brier = 0.1378, threshold = 0.3971
02:07:58 — Nested CV: outer fold 3/5
02:07:58 — Starting Optuna optimization for 100 trials
02:08:17 — Nested CV: fold 3 — test AUC = 0.8259, dev AUC = 0.8889, MCC = 0.4942, Brier = 0.1603, threshold = 0.3543
02:08:17 — Nested CV: outer fold 4/5
02:08:17 — Starting Optuna optimization for 100 trials
02:08:39 — Nested CV: fold 4 — test AUC = 0.8187, dev AUC = 0.9159, MCC = 0.4926, Brier = 0.2103, threshold = 0.3527
02:08:39 — Nested CV: outer fold 5/5
02:08:39 — Starting Optuna optimization for 100 trials
02:09:00 — Nested CV: fold 5 — test AUC = 0.8076, dev AUC = 0.9061, MCC 

Increasing `std_penalty` to `1.0` give some improvement:

- **Dev/test gap** closed significantly from `0.081` down to `0.058`, the best so thus
- **Dev std** halved from `0.0205` to `0.0127`, meaning the inner CV is much more stable

The trade-off is a slightly higher test std (`0.0236` --> `0.0266`), but this is minor

However, inspecting the number of trees across folds tells a different story - `[20, 64, 21, 10, 10]` where its mean is `25.0`. Fold 2 (`64`) deviates from the group,
while the remaining folds cluster around `10–21`, which is concerningly low. This suggests the model is unstable across folds, motivating a further increase in `std_penalty`.

In [121]:
nested_cv_frozen_nan_high_penatly['n_trees']

[20, 64, 21, 10, 10]

In [108]:
clf_frozen_nan_higher_penalty = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS, frozen_params={"gamma": 0, "alpha": 0}, std_penalty=2.0)

logger = logging.getLogger(__name__)
nested_cv_frozen_nan_higher_penalty = nested_cv_score(clf_frozen_nan_higher_penalty, X_nan, y, n_outer=5)

02:11:48 — Nested CV: outer fold 1/5
02:11:48 — Starting Optuna optimization for 100 trials
02:12:05 — Nested CV: fold 1 — test AUC = 0.8624, dev AUC = 0.8939, MCC = 0.5288, Brier = 0.1510, threshold = 0.3525
02:12:05 — Nested CV: outer fold 2/5
02:12:05 — Starting Optuna optimization for 100 trials
02:12:26 — Nested CV: fold 2 — test AUC = 0.8733, dev AUC = 0.8857, MCC = 0.5658, Brier = 0.1392, threshold = 0.4146
02:12:26 — Nested CV: outer fold 3/5
02:12:26 — Starting Optuna optimization for 100 trials
02:12:42 — Nested CV: fold 3 — test AUC = 0.8391, dev AUC = 0.8915, MCC = 0.5185, Brier = 0.1509, threshold = 0.3660
02:12:42 — Nested CV: outer fold 4/5
02:12:42 — Starting Optuna optimization for 100 trials
02:13:05 — Nested CV: fold 4 — test AUC = 0.8266, dev AUC = 0.9174, MCC = 0.4902, Brier = 0.1908, threshold = 0.3767
02:13:05 — Nested CV: outer fold 5/5
02:13:05 — Starting Optuna optimization for 100 trials
02:13:21 — Nested CV: fold 5 — test AUC = 0.8021, dev AUC = 0.8929, MCC 

The final configuration: `frozen_params={"gamma": 0, "alpha": 0}` along with `std_penalty=2.0` and zeros encoded as NaN

- **Highest test AUC**: `0.8407`, the best across all configurations
- **Smallest dev std**: `0.0109`, indicating a very stable inner CV

Compared to the default XGBoost benchmark (`0.7914`), the optimized pipeline delivers
a **+~0.04 AUC improvement**, confirming that the combination of:

1. Zero-to-NaN encoding to handle biologically implausible values.
2. Freezing low-importance hyperparameters (`gamma`, `alpha`) to focus the search

The number of trees across folds — `[35, 44, 34, 49, 35]` with a mean of `39.4` — is the most stable and consistent across all configurations. The abnormal effect observed
in our previous runs (single folds reaching `61–65`) has disappeared, with all folds clustering more tightly around `35–49`. This confirms that `std_penalty=2.0` may triger the model to find some more stable and consistent solutions across folds

In [109]:
nested_cv_frozen_nan_higher_penalty['n_trees']

[35, 44, 34, 49, 35]

In [110]:
pd.DataFrame(nested_cv_frozen_nan_higher_penalty['best_params'])

,lambda,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree
0,0.700323,0.061003,4,8,0.728217,0.680081
1,1.205894,0.090719,2,4,0.645390,0.702107
2,0.011837,0.109741,2,8,0.791995,0.698864
3,0.029372,0.009214,5,3,0.557209,0.657171
4,0.187830,0.028854,3,4,0.542908,0.662488


Despite the fact that `lambda` appeared unstable across folds, freezing it at its default value (`1`) cost some AUC and widened the dev/test gap slightly. Moreover, the number of trees seems unstable when `lambda = 1` -- meaning the hyperparameter seach is more unstable.

---

As a benchmark, a default `XGBClassifier` is by means of the same nested CV structure as `XGBOptClf`, but without any hyperparameter optimization. XGBoost's built-in defaults are used unmodified.

Two variants are evaluated:

- **No preprocessing** — zeros are left unmodified, passed directly to the model.
- **Zeros as NaN** 

In [145]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

def baseline_nested_cv(X, y, n_outer=N_FOLDS, random_state=RANDOM_STATE):
    """Nested CV for default XGBoost — mirrors nested_cv_score structure."""
    outer_cv = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state) # Note that StratifiedKFold
    scores, dev_aucs = [], []
    for dev_idx, test_idx in outer_cv.split(X, y):
        X_dev,  y_dev  = X[dev_idx],  y[dev_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        clf = XGBClassifier(random_state=random_state)
        clf.fit(X_dev, y_dev)
        dev_aucs.append(roc_auc_score(y_dev,  clf.predict_proba(X_dev)[:, 1]))
        scores.append(roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))
    return {"scores": np.array(scores), "dev_aucs": np.array(dev_aucs)}

# Run both combinations
baseline_no_prep = baseline_nested_cv(X, y)
baseline_nan     = baseline_nested_cv(X_nan, y)

# Summary
print("=" * 70)
print(f"{'Configuration':<35} {'Test AUC':<20} {'Dev AUC':<20}")
print("=" * 70)
print(f"{'No preprocessing':<35} {baseline_no_prep['scores'].mean():.4f} ± {baseline_no_prep['scores'].std():.4f}   {baseline_no_prep['dev_aucs'].mean():.4f} ± {baseline_no_prep['dev_aucs'].std():.4f}")
print(f"{'Zeros as NaN':<35} {baseline_nan['scores'].mean():.4f} ± {baseline_nan['scores'].std():.4f}   {baseline_nan['dev_aucs'].mean():.4f} ± {baseline_nan['dev_aucs'].std():.4f}")
print("=" * 70)

Configuration                       Test AUC             Dev AUC             
No preprocessing                    0.7914 ± 0.0284   1.0000 ± 0.0000
Zeros as NaN                        0.7947 ± 0.0327   1.0000 ± 0.0000


-------

## Experiment Tracking with MLflow

This cell logs the full experiment to MLflow, including:

- **Parameters** -- Optuna hyperparameters
- **Metrics** -- CV AUC mean/std, dev/test AUC, dev/test MCC, applied threshold, nested CV scores per fold
- **Artifacts** -- classification reports (JSON), ROC curve (PNG), Optuna plots (HTML), trained XGBoost model

### Nested CV
Performance is obtained via nested cross-validation (`n_outer=5`, `std_penalty=1`).
The outer loop evaluates performance; the inner loop (inside `fit()`) optimises hyperparameters via Optuna.

### Viewing Results
Launch the MLflow UI with:
```bash
cd notebooks

mlflow ui --port 5002
```
Then open `http://localhost:5002` in your browser.

In [ ]:
pd.DataFrame(nested_cv_frozen_nan_higher_penalty['best_params'])

,lambda,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree
0,0.700323,0.061003,4,8,0.728217,0.680081
1,1.205894,0.090719,2,4,0.645390,0.702107
2,0.011837,0.109741,2,8,0.791995,0.698864
3,0.029372,0.009214,5,3,0.557209,0.657171
4,0.187830,0.028854,3,4,0.542908,0.662488


In [112]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X_nan, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

`Insulin` (`47.23%`) and `SkinThickness` (`28.50%`) have *significant* missing rates, which XGBoost handles natively by learning the optimal split direction for missing values.
The importance  `min_child_weight`  is consistent with this phenomen; it guards against splits driven by too few observed values.

In [117]:
import pandas as pd

# count NaNs per feature
nan_counts = pd.DataFrame(X_nan, columns=feature_cols).isna().sum()
nan_pct    = pd.DataFrame(X_nan, columns=feature_cols).isna().mean() * 100

summary = pd.DataFrame({
    "NaN count": nan_counts,
    "NaN %":     nan_pct.round(2)
})

print(summary[summary["NaN count"] > 0])

               NaN count  NaN %
Glucose                5   0.65
BloodPressure         35   4.56
SkinThickness        227  29.56
Insulin              374  48.70
BMI                   11   1.43


In [118]:
import pandas as pd

# count NaNs per feature
nan_counts = pd.DataFrame(X_dev, columns=feature_cols).isna().sum()
nan_pct    = pd.DataFrame(X_dev, columns=feature_cols).isna().mean() * 100

summary = pd.DataFrame({
    "NaN count": nan_counts,
    "NaN %":     nan_pct.round(2)
})

print(summary[summary["NaN count"] > 0])

               NaN count  NaN %
Glucose                4   0.65
BloodPressure         23   3.75
SkinThickness        175  28.50
Insulin              290  47.23
BMI                    9   1.47


In [ ]:
import os, json, time, tempfile
import mlflow
import optuna.visualization as vis
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve


mlflow.set_tracking_uri(f"file://{os.path.abspath('mlruns')}")
mlflow.set_experiment(STUDY_NAME)

with mlflow.start_run(run_name="pima_diabetes"):

    # ─── Tags 
    mlflow.set_tags({
        "model":   "XGBOptClf",
        "dataset": "pima_diabetes_no_preprocessing",
    })

    # ─── Fit 
    clf = XGBOptClf(n_trials=N_TRIALS, nfold=N_FOLDS, frozen_params={"gamma": 0, "alpha": 0}, std_penalty=2.0)
    clf.fit(X_dev, y_dev)

    # ─── Params 
    mlflow.log_params({
        **clf.best_params_,
        "n_trials":    clf.n_trials,
        "n_estimators": clf.best_num_boost_round_,
        "n_jobs":      clf._resolve_n_jobs(),
        "std_penalty": clf.std_penalty,
        "eval_metric": clf.eval_metric,
        "nfold":       clf.nfold,
    })
    
    # ─── Eval Metrics 
    results = clf.eval(X_dev, y_dev, X_test, y_test)
    mlflow.log_metrics({
        "dev_auc":           results["dev_auc"],
        "dev_mcc":           results["dev_mcc"],
        "test_auc":          results["test_auc"],
        "test_mcc":          results["test_mcc"],
        "applied_threshold": results["applied_threshold"],
        "dev_brier":        results["dev_brier"],
        "test_brier":       results["test_brier"],
    })

    # ─── Nested CV 
    nested_cv = nested_cv_score(clf, X_nan, y, n_outer=5, stratified=True)
    mlflow.log_metrics({
        "nested_cv_test_mean_auc": nested_cv["scores"].mean(),
        "nested_cv_test_std_auc":  nested_cv["scores"].std(),
        "nested_cv_dev_mean_auc":  nested_cv["dev_aucs"].mean(),
        "nested_cv_dev_std_auc":   nested_cv["dev_aucs"].std(),
        "nested_cv_test_mean_mcc":      nested_cv["test_mccs"].mean(),
        "nested_cv_test_std_mcc":       nested_cv["test_mccs"].std(),
        "nested_cv_dev_mean_mcc":      nested_cv["dev_mccs"].mean(),
        "nested_cv_dev_std_mcc":       nested_cv["dev_mccs"].std(),
        "nested_cv_mean_threshold": nested_cv["optimal_thresholds"].mean(),
        "nested_cv_std_threshold":  nested_cv["optimal_thresholds"].std(),
    })

    # ─── Nested CV per fold 
    for i, (score, mcc, threshold, n_tree, params) in enumerate(zip(
        nested_cv["scores"],
        nested_cv["test_mccs"],
        nested_cv["optimal_thresholds"],
        nested_cv["n_trees"],
        nested_cv["best_params"]
    )):
        mlflow.log_metrics({
            f"fold_{i+1}_auc":       score,
            f"fold_{i+1}_mcc":       mcc,
            f"fold_{i+1}_threshold": threshold,
            f"fold_{i+1}_n_trees":   n_tree,
        })
        mlflow.log_dict(params, f"fold_{i+1}_params.json")

    with tempfile.TemporaryDirectory() as tmp:

        # ─── Thresholds & MCC across folds 
        threshold_df = pd.DataFrame({
            "fold":      range(1, len(nested_cv["optimal_thresholds"]) + 1),
            "threshold": nested_cv["optimal_thresholds"],
            "auc":       nested_cv["scores"],
            "mcc":       nested_cv["test_mccs"],
        })
        threshold_df.loc["mean"] = ["mean",
                                     nested_cv["optimal_thresholds"].mean(),
                                     nested_cv["scores"].mean(),
                                     nested_cv["test_mccs"].mean()]
        threshold_df.loc["std"]  = ["std",
                                     nested_cv["optimal_thresholds"].std(),
                                     nested_cv["scores"].std(),
                                     nested_cv["test_mccs"].std()]
        print(threshold_df.to_string(index=False))
        path = os.path.join(tmp, "thresholds_across_folds.csv")
        threshold_df.to_csv(path, index=False)
        mlflow.log_artifact(path)

        # ─── ROC Curve 
        fpr, tpr, thresholds = roc_curve(y_test, clf.predict_proba(X_test)[:, 1])
        idx = np.argmin(np.abs(thresholds - results["applied_threshold"]))
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f"AUC = {clf.score(X_test, y_test):.4f}", linewidth=2)
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.scatter(fpr[idx], tpr[idx], color="red", zorder=5,
                label=f"Applied Threshold = {results['applied_threshold']:.2f}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve")
        ax.legend()
        fig.savefig(os.path.join(tmp, "roc_curve.png"))
        mlflow.log_artifact(os.path.join(tmp, "roc_curve.png"))
        plt.close(fig)

        # ─── Optuna Plots 
        fig1, fig2, fig3, fig4 = clf.plot_optuna_insights()
        for name, fig in [
            ("optimization_history", fig1),
            ("param_importances",    fig2),
            ("slice",                fig3),
            ("parallel_coordinate",  fig4),
        ]:
            path = os.path.join(tmp, f"{name}.html")
            fig.write_html(path)
            mlflow.log_artifact(path)

        # ─── Log Model ────────────────────────────────────────────────────────
        mlflow.xgboost.log_model(clf.best_model_, artifact_path="model")

10:52:18 — Starting Optuna optimization for 100 trials
10:52:33 — Nested CV: outer fold 1/5
10:52:33 — Starting Optuna optimization for 100 trials
10:52:49 — Nested CV: fold 1 — test AUC = 0.8624, dev AUC = 0.8939, MCC = 0.5288, Brier = 0.1510, threshold = 0.3525
10:52:49 — Nested CV: outer fold 2/5
10:52:49 — Starting Optuna optimization for 100 trials
10:53:08 — Nested CV: fold 2 — test AUC = 0.8733, dev AUC = 0.8857, MCC = 0.5658, Brier = 0.1392, threshold = 0.4146
10:53:08 — Nested CV: outer fold 3/5
10:53:08 — Starting Optuna optimization for 100 trials
10:53:21 — Nested CV: fold 3 — test AUC = 0.8391, dev AUC = 0.8915, MCC = 0.5185, Brier = 0.1509, threshold = 0.3660
10:53:21 — Nested CV: outer fold 4/5
10:53:21 — Starting Optuna optimization for 100 trials
10:53:41 — Nested CV: fold 4 — test AUC = 0.8266, dev AUC = 0.9174, MCC = 0.4902, Brier = 0.1908, threshold = 0.3767
10:53:41 — Nested CV: outer fold 5/5
10:53:41 — Starting Optuna optimization for 100 trials
10:53:57 — Nested

fold  threshold      auc      mcc
   1   0.352488 0.862407 0.528786
   2   0.414594 0.873333 0.565761
   3   0.365964 0.839074 0.518545
   4   0.376682 0.826604 0.490222
   5   0.348616 0.802075 0.420425
mean   0.371669 0.840699 0.504748
 std   0.023663 0.025414 0.048605


2026/05/12 10:54:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/12 10:54:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
